# Orquestador Central: Pipeline Palta Inteligente
Este Notebook actúa como el panel de control interactivo para ejecutar, validar y auditar el pipeline ETL completo.

### Paso 1: Inicializar el entorno y variables de entorno

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import sqlite3

# 1. Detectamos las rutas
ruta_notebook = Path(os.getcwd())
ruta_raiz = ruta_notebook.parent # Sube un nivel a la raíz del proyecto

# 2. Cambiamos el directorio de trabajo de Python a la raíz
os.chdir(ruta_raiz)
print(f"Directorio de trabajo establecido en la raíz: {os.getcwd()}")

# 3. Cargamos el .env directamente desde la raíz
load_dotenv(override=True)

resource_id = os.getenv("ODEPA_RESOURCE_ID")
if resource_id:
    print("Variables de entorno cargadas con éxito.")
    print("ID de Recurso ODEPA:", resource_id[:6] + "...")
else:
    print("ERROR: No se pudo cargar 'ODEPA_RESOURCE_ID'.")

Directorio de trabajo establecido en la raíz: c:\Users\aguir\Paltas_rentables
Variables de entorno cargadas con éxito.
ID de Recurso ODEPA: 580bec...


### Paso 2: Ejecutar e Inicializar la Base de Datos Relacional

In [18]:
print("--- [1/4] Inicializando Base de Datos ---")
%run etl/setup_db.py

conexion = sqlite3.connect("data/paltas_retail.db")
df_reglas = pd.read_sql_query("SELECT * FROM reglas_fenologicas", conexion)
conexion.close()
display(df_reglas)

--- [1/4] Inicializando Base de Datos ---


2026-06-15 02:41:15,688 - [INFO] - Creando base de datos relacional en: data\paltas_retail.db
2026-06-15 02:41:15,694 - [INFO] - Poblando tablas con parámetros de negocio...
2026-06-15 02:41:15,697 - [INFO] - ¡Base de datos SQL inicializada con éxito!


,id_regla,fase_planta,mes_clima,temp_minima_critica,mes_impacto_mercado
0,1,Floración (Invierno),8,2.0,4
1,2,Desarrollo de Fruto,11,5.0,5


### Paso 3: Extracción de Clima Histórico (Open-Meteo)

In [20]:
print("--- [2/4] Ejecutando ETL Clima ---")
%run etl/ETL_clima.py 

ruta_verificacion_clima = "data/historical_weather.csv" 

if os.path.exists(ruta_verificacion_clima):
    df_clima = pd.read_csv(ruta_verificacion_clima)
    print(f"\nCSV de clima generado exitosamente.")
    print(f"Total de filas detectadas: {df_clima.shape[0]}")
    display(df_clima.head(3))

2026-06-15 02:41:33,341 - [INFO] - Iniciando descarga masiva de clima histórico...


--- [2/4] Ejecutando ETL Clima ---


2026-06-15 02:41:34,559 - [INFO] - ¡Éxito! CSV de clima generado perfectamente en: data\historical_weather.csv


  fecha_clima      region  temp_max  temp_min  lluvia_mm
0  2023-01-01  VALPARAISO      25.6      15.7        0.0
1  2023-01-02  VALPARAISO      24.3      15.5        0.0
2  2023-01-03  VALPARAISO      27.1      15.4        0.0
3  2023-01-04  VALPARAISO      27.0      19.3        0.0
4  2023-01-05  VALPARAISO      25.5      15.5        0.0

CSV de clima generado exitosamente.
Total de filas detectadas: 3774


,fecha_clima,region,temp_max,temp_min,lluvia_mm
0,2023-01-01,VALPARAISO,25.6,15.7,0.0
1,2023-01-02,VALPARAISO,24.3,15.5,0.0
2,2023-01-03,VALPARAISO,27.1,15.4,0.0


### Paso 4: Extracción de Precios Mayoristas (ODEPA + Pydantic)

In [21]:
print("--- [3/4] Ejecutando ETL ODEPA ---")
%run etl/ETL_odepa.py

# Verificación del archivo generado
if os.path.exists("data/raw_odepa_paltas.csv"):
    df_odepa = pd.read_csv("data/raw_odepa_paltas.csv")
    print(f"\nCSV de ODEPA generado (Sólo Paltas). Filas: {df_odepa.shape[0]}")
    display(df_odepa.head(3))

--- [3/4] Ejecutando ETL ODEPA ---


2026-06-15 02:41:53,716 - [INFO] - --- INICIANDO EXTRACCIÓN ODEPA ---
2026-06-15 02:41:53,717 - [INFO] - Iniciando extracción masiva desde la API RESTful de ODEPA...
2026-06-15 02:41:54,473 - [INFO] - Se extrajeron 10000 registros crudos de todo tipo de productos.
2026-06-15 02:41:54,567 - [INFO] - Filtrando localmente solo los registros de Palta...
2026-06-15 02:41:54,592 - [INFO] - ¡Éxito! Se guardaron 576 registros de Palta en: data\raw_odepa_paltas.csv



CSV de ODEPA generado (Sólo Paltas). Filas: 576


,id_registro,fecha,producto,mercado,precio_minimo,precio_maximo,precio_promedio
0,342,2026-01-02,Palta,Vega Monumental Concepción,3600.0,3600.0,3600.0
1,343,2026-01-02,Palta,Vega Monumental Concepción,3300.0,3300.0,3300.0
2,344,2026-01-02,Palta,Vega Monumental Concepción,2800.0,2800.0,2800.0


### Paso 5: Unificación y Generación del Dataset Maestro (Lag Features)

In [22]:
print("--- [4/4] Ejecutando Transformación Master ---")
%run etl/ETL_master.py

# Auditoría final del entregable
df_master = pd.read_csv("data/master_dataset.csv")
print(f"\n¡Pipeline Exitoso! Dataset final unificado para el Dashboard.")
print(f"Total registros: {df_master.shape[0]}")
print(f"Alertas de heladas previas detectadas: {df_master['alerta_helada_previa'].sum()}")
display(df_master.tail(5))

2026-06-15 02:42:20,893 - [INFO] - Iniciando Fase de Transformación y Carga (T & L)...
2026-06-15 02:42:20,906 - [INFO] - Aplicando lógica de rezago agrícola (8 meses)...
2026-06-15 02:42:20,951 - [INFO] - Pipeline ETL completado. Tabla maestra generada en: data\master_dataset.csv
2026-06-15 02:42:20,952 - [INFO] - Total de registros unificados: 576


--- [4/4] Ejecutando Transformación Master ---

¡Pipeline Exitoso! Dataset final unificado para el Dashboard.
Total registros: 576
Alertas de heladas previas detectadas: 576


,id_registro,fecha,producto,mercado,precio_minimo,precio_maximo,precio_promedio,temp_min,temp_max,lluvia_mm,alerta_helada_previa
571,9843,2026-01-20,Palta,Mercado Mayorista Lo Valledor de Santiago,2200.0,2200.0,2200.0,2.0,31.3,161.3,1
572,9844,2026-01-20,Palta,Mercado Mayorista Lo Valledor de Santiago,3200.0,3200.0,3200.0,2.0,31.3,161.3,1
573,9845,2026-01-20,Palta,Mercado Mayorista Lo Valledor de Santiago,2000.0,2000.0,2000.0,2.0,31.3,161.3,1
574,9846,2026-01-20,Palta,Mercado Mayorista Lo Valledor de Santiago,2800.0,2800.0,2800.0,2.0,31.3,161.3,1
575,9847,2026-01-20,Palta,Mercado Mayorista Lo Valledor de Santiago,2500.0,2500.0,2500.0,2.0,31.3,161.3,1


### Celda Final: Lanzar el Dashboard Interactivo desde el Notebook

In [4]:
print("Levantando el servidor de Streamlit...")
print("Se abrirá una pestaña en tu navegador. Para apagar el servidor, dale al botón 'Stop' en Jupyter.")

!streamlit run dashboards/dashboard_code.py

Levantando el servidor de Streamlit...
Se abrirá una pestaña en tu navegador. Para apagar el servidor, dale al botón 'Stop' en Jupyter.
^C
